# 第 10 章：Human-in-the-loop——Interrupt、恢复、Time Travel 与副作用安全（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch10-approve-restart`

In [3]:
from pathlib import Path
import tempfile
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
from mini_deerflow.graph import create_approval_workflow
from mini_deerflow.persistence import SqliteEffectLedger

with tempfile.TemporaryDirectory() as approval_directory:
    approval_root = Path(approval_directory)
    approval_checkpoint_path = approval_root / "checkpoints.sqlite"
    approval_effects = SqliteEffectLedger(approval_root / "effects.sqlite")
    approval_config = {"configurable": {"thread_id": "publish-001"}}
    approval_request = {
        "request_id": "publish-001",
        "action": "publish_report",
        "payload": {"path": "reports/final.md"},
        "review_stages": ["risk"],
    }

    with SqliteSaver.from_conn_string(str(approval_checkpoint_path)) as saver:
        approval_graph = create_approval_workflow(
            checkpointer=saver,
            effect_ledger=approval_effects,
        )
        paused_approval = approval_graph.invoke(
            approval_request,
            config=approval_config,
        )
        assert paused_approval["__interrupt__"][0].value["stage"] == "risk"
        assert approval_graph.get_state(approval_config).next == ("review",)

    with SqliteSaver.from_conn_string(str(approval_checkpoint_path)) as reopened_saver:
        restarted_approval_graph = create_approval_workflow(
            checkpointer=reopened_saver,
            effect_ledger=approval_effects,
        )
        approved_result = restarted_approval_graph.invoke(
            Command(resume={"decision": "approve"}),
            config=approval_config,
        )

    assert approved_result["status"] == "completed"
    assert approved_result["effect_status"] == "recorded"
    assert approval_effects.count("publish-001") == 1


### 实验 `ch10-edit-reject`

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

with tempfile.TemporaryDirectory() as decision_directory:
    decision_effects = SqliteEffectLedger(Path(decision_directory) / "effects.sqlite")
    decision_graph = create_approval_workflow(
        checkpointer=InMemorySaver(),
        effect_ledger=decision_effects,
    )

    edit_config = {"configurable": {"thread_id": "edit-001"}}
    decision_graph.invoke(
        {
            "request_id": "edit-001",
            "action": "publish_report",
            "payload": {"path": "reports/draft.md"},
        },
        config=edit_config,
    )
    edited_result = decision_graph.invoke(
        Command(
            resume={
                "decision": "edit",
                "edited_payload": {"path": "reports/reviewed.md"},
            }
        ),
        config=edit_config,
    )
    assert edited_result["payload"] == {"path": "reports/reviewed.md"}
    assert decision_effects.count("edit-001") == 1

    reject_config = {"configurable": {"thread_id": "reject-001"}}
    decision_graph.invoke(
        {
            "request_id": "reject-001",
            "action": "publish_report",
            "payload": {"path": "reports/unsafe.md"},
        },
        config=reject_config,
    )
    rejected_result = decision_graph.invoke(
        Command(resume={"decision": "reject", "reason": "证据不足"}),
        config=reject_config,
    )
    assert rejected_result["status"] == "rejected"
    assert decision_effects.count("reject-001") == 0


### 实验 `ch10-time-travel-idempotency`

In [5]:
with tempfile.TemporaryDirectory() as replay_directory:
    replay_effects = SqliteEffectLedger(Path(replay_directory) / "effects.sqlite")
    replay_graph = create_approval_workflow(
        checkpointer=InMemorySaver(),
        effect_ledger=replay_effects,
    )
    replay_config = {"configurable": {"thread_id": "replay-001"}}
    replay_graph.invoke(
        {
            "request_id": "replay-001",
            "action": "publish_report",
            "payload": {"path": "reports/final.md"},
        },
        config=replay_config,
    )
    replay_graph.invoke(
        Command(resume={"decision": "approve"}),
        config=replay_config,
    )
    before_effect = next(
        snapshot
        for snapshot in replay_graph.get_state_history(replay_config)
        if snapshot.next == ("record_effect_intent",)
    )

    replayed_result = replay_graph.invoke(None, config=before_effect.config)

    assert replayed_result["effect_status"] == "already_recorded"
    assert replay_effects.count("replay-001") == 1


Deserializing unregistered type mini_deerflow.graph.approval.ApprovalDecision from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('mini_deerflow.graph.approval', 'ApprovalDecision')]


### 实验 `ch10-concurrent-intent`

In [6]:
from concurrent.futures import ThreadPoolExecutor
import threading

with tempfile.TemporaryDirectory() as concurrent_directory:
    concurrent_path = Path(concurrent_directory) / "effects.sqlite"
    concurrent_ledgers = [
        SqliteEffectLedger(concurrent_path),
        SqliteEffectLedger(concurrent_path),
    ]
    concurrent_barrier = threading.Barrier(2)

    def record_concurrently(ledger: SqliteEffectLedger) -> str:
        concurrent_barrier.wait()
        return ledger.record_once(
            "concurrent-001",
            "publish_report",
            {"path": "reports/final.md"},
        ).status

    with ThreadPoolExecutor(max_workers=2) as executor:
        concurrent_statuses = list(
            executor.map(record_concurrently, concurrent_ledgers)
        )

    assert sorted(concurrent_statuses) == ["already_recorded", "recorded"]
    assert concurrent_ledgers[0].count("concurrent-001") == 1


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

### 实验 `ch10-multiple-interrupt-events`

In [7]:
with tempfile.TemporaryDirectory() as multi_directory:
    multi_effects = SqliteEffectLedger(Path(multi_directory) / "effects.sqlite")
    multi_graph = create_approval_workflow(
        checkpointer=InMemorySaver(),
        effect_ledger=multi_effects,
    )
    multi_config = {"configurable": {"thread_id": "multi-001"}}
    multi_request = {
        "request_id": "multi-001",
        "action": "publish_report",
        "payload": {"path": "reports/final.md"},
        "review_stages": ["risk", "compliance"],
    }

    first_entry_events = list(
        multi_graph.stream(multi_request, config=multi_config, stream_mode="custom")
    )
    first_stage = multi_graph.get_state(multi_config).tasks[0].interrupts[0].value["stage"]
    second_entry_events = list(
        multi_graph.stream(
            Command(resume={"decision": "approve"}),
            config=multi_config,
            stream_mode="custom",
        )
    )
    second_stage = multi_graph.get_state(multi_config).tasks[0].interrupts[0].value["stage"]
    final_entry_events = list(
        multi_graph.stream(
            Command(resume={"decision": "approve"}),
            config=multi_config,
            stream_mode="custom",
        )
    )

    assert (first_stage, second_stage) == ("risk", "compliance")
    assert [
        first_entry_events[0]["event"],
        second_entry_events[0]["event"],
        final_entry_events[0]["event"],
    ] == ["review_node_entered"] * 3
    assert multi_graph.get_state(multi_config).values["status"] == "completed"


## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch10-side-effect-before-interrupt-failure`

In [8]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.types import interrupt

class UnsafeApprovalState(TypedDict):
    request_id: str

unsafe_effects: list[str] = []

def unsafe_review(state: UnsafeApprovalState) -> dict[str, str]:
    unsafe_effects.append(state["request_id"])
    interrupt({"request_id": state["request_id"]})
    return {}

unsafe_builder = StateGraph(UnsafeApprovalState)
unsafe_builder.add_node("review", unsafe_review)
unsafe_builder.add_edge(START, "review")
unsafe_builder.add_edge("review", END)
unsafe_graph = unsafe_builder.compile(checkpointer=InMemorySaver())
unsafe_config = {"configurable": {"thread_id": "unsafe-001"}}

unsafe_graph.invoke({"request_id": "unsafe-001"}, config=unsafe_config)
unsafe_graph.invoke(Command(resume="approve"), config=unsafe_config)

assert unsafe_effects == ["unsafe-001", "unsafe-001"]


### 实验 `ch10-idempotency-conflict-failure`

In [9]:
from mini_deerflow.persistence import IdempotencyConflictError

with tempfile.TemporaryDirectory() as conflict_directory:
    conflict_ledger = SqliteEffectLedger(Path(conflict_directory) / "effects.sqlite")
    conflict_ledger.record_once(
        "operation-001",
        "publish_report",
        {"path": "reports/final.md"},
    )
    try:
        conflict_ledger.record_once(
            "operation-001",
            "delete_report",
            {"path": "reports/final.md"},
        )
    except IdempotencyConflictError as error:
        conflict_error = error
    else:
        raise AssertionError("相同 idempotency key 的不同副作用必须失败")

assert "不同副作用" in str(conflict_error)


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。